In [ ]:
!pip install ultralytics


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 67.5 MB/s eta 0:00:00


In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from tqdm import tqdm
from ultralytics import YOLO

DATA_DIR = "/content/lab6"
CLASS_NAMES = ["inaction", "move", "work"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Инициализируем экстракторы
mobilenet_global = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT).to(device).eval()
mobilenet_global.classifier = nn.Identity()

mobilenet_local = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT).to(device).eval()
mobilenet_local.classifier = nn.Identity()

yolo_detector = YOLO("yolov8m-pose.pt")

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def extract_super_hybrid_features(frames_paths):
    sequence_features = []
    prev_gray = None

    for pth in frames_paths:
        frame = cv2.imread(pth)
        if frame is None:
            sequence_features.append(np.zeros(960 + 960 + 3))
            continue

        frame = cv2.resize(frame, (640, 480))
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # 1. Глобальный эмбеддинг (весь кадр)
        pil_global = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        tensor_global = preprocess(pil_global).unsqueeze(0).to(device)
        with torch.no_grad():
            feat_global = mobilenet_global(tensor_global).squeeze().cpu().numpy()

        # 2. Локальный эмбеддинг (вырезаем человека)
        results = yolo_detector(frame, verbose=False)

        # ИСПРАВЛЕНИЕ: Достаем первый результат из списка результатов YOLO
        res = results[0]

        bx1, by1, bx2, by2 = 0, 0, 640, 480
        motion_score, flow_x, flow_y = 0.0, 0.0, 0.0

        if len(res.boxes) > 0:
            max_area = 0
            for box in res.boxes:
                if int(box.cls) == 0:  # Только класс Person
                    coords = box.xyxy.cpu().numpy().astype(int).squeeze()
                    if coords.ndim == 1 and len(coords) == 4:
                        x1, y1, x2, y2 = coords
                        area = (x2 - x1) * (y2 - y1)
                        if area > max_area:
                            max_area = area
                            bx1, by1, bx2, by2 = x1, y1, x2, y2

        bx1, by1, bx2, by2 = max(0, bx1), max(0, by1), min(640, bx2), min(480, by2)

        person_crop = frame[by1:by2, bx1:bx2]
        if person_crop.size == 0:
            person_crop = frame

        pil_local = Image.fromarray(cv2.cvtColor(person_crop, cv2.COLOR_BGR2RGB))
        tensor_local = preprocess(pil_local).unsqueeze(0).to(device)
        with torch.no_grad():
            feat_local = mobilenet_local(tensor_local).squeeze().cpu().numpy()

        # 3. Движение внутри силуэта
        if prev_gray is not None:
            crop_curr = gray[by1:by2, bx1:bx2]
            crop_prev = prev_gray[by1:by2, bx1:bx2]
            if crop_curr.size > 0 and crop_curr.shape == crop_prev.shape:
                motion_score = np.mean(cv2.absdiff(crop_curr, crop_prev) > 20)
                flow = cv2.calcOpticalFlowFarneback(crop_prev, crop_curr, None, 0.5, 3, 15, 3, 5, 1.2, 0)
                flow_x, flow_y = np.mean(flow[..., 0]), np.mean(flow[..., 1])

        prev_gray = gray

        # Конкатенируем всё вместе: 960 + 960 + 3 = 1923 фичи
        combined_feat = np.concatenate([feat_global, feat_local, [motion_score, flow_x, flow_y]])
        sequence_features.append(combined_feat)

    return np.array(sequence_features)


# Сбор данных
X_data, y_data = [], []
for class_idx, class_name in enumerate(CLASS_NAMES):
    class_folder = os.path.join(DATA_DIR, class_name)
    if not os.path.exists(class_folder): continue
    print(f"\n🚀 Сборка СУПЕР-датасета для класса: {class_name}")
    subfolders = [os.path.join(class_folder, d) for d in os.listdir(class_folder) if os.path.isdir(os.path.join(class_folder, d))]
    for subfolder in tqdm(subfolders):
        images = sorted([os.path.join(subfolder, f) for f in os.listdir(subfolder) if f.lower().endswith(('.png', '.jpg', '.jpeg')) and not f.startswith('.')])
        if len(images) == 0: continue
        while len(images) < 8: images.append(images[-1])
        images = images[:8]
        X_data.append(extract_super_hybrid_features(images))
        y_data.append(class_idx)

np.save("X_super_features.npy", np.array(X_data))
np.save("y_super_labels.npy", np.array(y_data))
print(f"\n🎉 Датасет-комбо готов! Новая размерность: {np.array(X_data).shape}")



🚀 Сборка СУПЕР-датасета для класса: inaction


100%|██████████| 90/90 [01:18<00:00,  1.14it/s]



🚀 Сборка СУПЕР-датасета для класса: move


100%|██████████| 150/150 [02:13<00:00,  1.13it/s]



🚀 Сборка СУПЕР-датасета для класса: work


100%|██████████| 342/342 [05:02<00:00,  1.13it/s]


🎉 Датасет-комбо готов! Новая размерность: (582, 8, 1923)


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. ЗАГРУЗКА
X = np.load("X_super_features.npy")
y = np.load("y_super_labels.npy")
CLASS_NAMES = ["inaction", "move", "work"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

X_train_t = torch.tensor(X_train, dtype=torch.float32).transpose(1, 2)
X_val_t = torch.tensor(X_val, dtype=torch.float32).transpose(1, 2)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=16, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=16, shuffle=False)

# 2. АРХИТЕКТУРА ПОД СУПЕР-ФИЧИ
class SuperTemporalConvNet(nn.Module):
    def __init__(self, in_channels=1923, num_classes=3):
        super(SuperTemporalConvNet, self).__init__()
        self.conv1 = nn.Conv1d(in_channels, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.6)

        self.conv2 = nn.Conv1d(64, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(32)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.6)

        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(32, num_classes)

    def forward(self, x):
        x = self.dropout1(self.relu1(self.bn1(self.conv1(x))))
        x = self.dropout2(self.relu2(self.bn2(self.conv2(x))))
        x = self.pool(x).squeeze(-1)
        return self.fc(x)

model = SuperTemporalConvNet().to(device)
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr=0.0002, weight_decay=1e-2)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4)

# 3. ОБУЧЕНИЕ
NUM_EPOCHS = 50
best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    model.train()
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item() * batch_X.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += batch_y.size(0)
            correct_val += (predicted == batch_y).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())

    val_loss_avg = val_loss / total_val
    scheduler.step(val_loss_avg)

    if val_loss_avg < best_val_loss:
        best_val_loss = val_loss_avg
        torch.save(model.state_dict(), "best_super_cnn.pth")

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Эпоха [{epoch+1}/{NUM_EPOCHS}] | Loss Val: {val_loss_avg:.4f} | Acc Val: {correct_val/total_val:.2%}")

print("\n📋 ИТОГОВЫЙ СУПЕР-ОТЧЕТ КОМБО-МОДЕЛИ:")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))


Эпоха [1/50] | Loss Val: 1.0104 | Acc Val: 56.41%
Эпоха [10/50] | Loss Val: 0.6941 | Acc Val: 77.78%
Эпоха [20/50] | Loss Val: 0.5724 | Acc Val: 81.20%
Эпоха [30/50] | Loss Val: 0.5760 | Acc Val: 79.49%
Эпоха [40/50] | Loss Val: 0.5739 | Acc Val: 79.49%
Эпоха [50/50] | Loss Val: 0.5346 | Acc Val: 82.05%

📋 ИТОГОВЫЙ СУПЕР-ОТЧЕТ КОМБО-МОДЕЛИ:
              precision    recall  f1-score   support

    inaction       0.69      0.61      0.65        18
        move       0.73      0.80      0.76        30
        work       0.90      0.88      0.89        69

    accuracy                           0.82       117
   macro avg       0.77      0.77      0.77       117
weighted avg       0.82      0.82      0.82       117



In [ ]:
import os
import glob
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import cv2
import numpy as np
import random
from torchvision.models.video import r3d_18, R3D_18_Weights
from sklearn.utils.class_weight import compute_class_weight

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используем устройство для защищенной 3D-ResNet18: {device}")

# ==========================================
# 1. УЛУЧШЕННЫЙ ДАТАСЕТ С АУГМЕНТАЦИЕЙ ВИДЕО
# ==========================================
class AugmentedVideoDataset(Dataset):
    def __init__(self, data_dir, clip_len=16, is_train=True):
        self.data_dir = data_dir
        self.clip_len = clip_len
        self.is_train = is_train
        self.classes = ["inaction", "move", "work"]
        self.samples = []
        self.labels_all = [] # Для подсчета весов классов

        valid_exts = ('*.png', '*.jpg', '*.jpeg', '*.bmp', '*.webp')

        for class_idx, class_name in enumerate(self.classes):
            class_folder = os.path.join(data_dir, class_name)
            if not os.path.exists(class_folder): continue

            subfolders = [os.path.join(class_folder, d) for d in os.listdir(class_folder) if os.path.isdir(os.path.join(class_folder, d))]

            for subfolder in subfolders:
                images = []
                for ext in valid_exts:
                    images.extend(glob.glob(os.path.join(subfolder, ext)))
                images = sorted(images)

                if len(images) == 0: continue

                if len(images) < clip_len:
                    while len(images) < clip_len:
                        images.append(images[-1])
                    self.samples.append((images, class_idx))
                    self.labels_all.append(class_idx)
                else:
                    stride = 16
                    for start_idx in range(0, len(images) - clip_len + 1, stride):
                        clip_paths = images[start_idx : start_idx + clip_len]
                        self.samples.append((clip_paths, class_idx))
                        self.labels_all.append(class_idx)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_paths, label = self.samples[idx]
        frames = []

        # Аугментация: выбираем параметры один раз на весь клип из 16 кадров (для сохранения тайминга)
        do_flip = self.is_train and (random.random() > 0.5)

        for pth in img_paths:
            img = cv2.imread(pth)
            if img is None:
                img = np.zeros((112, 112, 3), dtype=np.uint8)
            else:
                img = cv2.resize(img, (112, 112))
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # Эффект отзеркаливания (аугментация направления движения)
            if do_flip:
                img = cv2.flip(img, 1)
            frames.append(img)

        video = np.array(frames, dtype=np.float32) / 255.0
        video = np.transpose(video, (3, 0, 1, 2)) # (Channels, Time, H, W)

        # Нормализация ImageNet
        mean = np.array([0.485, 0.456, 0.406])[:, None, None, None]
        std = np.array([0.229, 0.224, 0.225])[:, None, None, None]
        video = (video - mean) / std

        return torch.tensor(video, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

# ==========================================
# 2. ИНИЦИАЛИЗАЦИЯ ДАННЫХ И ПОДСЧЕТ ВЕСОВ КЛАССОВ
# ==========================================
DATA_DIR = "/content/lab6"

# Создаем полный датасет БЕЗ аугментации для разделения,
# флаг is_train переключим динамически, чтобы не аугментировать валидацию
full_dataset = AugmentedVideoDataset(DATA_DIR, clip_len=16, is_train=False)

# Считаем сбалансированные веса классов против дисбаланса (класса work больше)
y_all = np.array(full_dataset.labels_all)
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_all), y=y_all)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"🧬 Рассчитанные веса для балансировки Loss функции: {class_weights}")

# Разбиваем на Train/Val (80/20)
torch.manual_seed(42)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Принудительно включаем аугментацию для тренировочной части
train_dataset.dataset.is_train = True

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# ==========================================
# 3. НАСТРОЙКА МОДЕЛИ С DROPOUT В КЛАССИФИКАТОРЕ
# ==========================================
print("📥 Загрузка базовой r3d_18...")
model_3d = r3d_18(weights=R3D_18_Weights.DEFAULT)

# Модифицируем финальный слой: добавляем Dropout(0.5) перед линейным классификатором
in_features = model_3d.fc.in_features
model_3d.fc = nn.Sequential(
    nn.Dropout(p=0.5), # Защита классификатора от переобучения
    nn.Linear(in_features, 3)
)
model_3d = model_3d.to(device)

# Передаем сбалансированные веса классов в функцию потерь!
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.AdamW(model_3d.parameters(), lr=1e-4, weight_decay=1e-2) # Усилен weight_decay
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# ==========================================
# 4. ЦИКЛ ОБУЧЕНИЯ С EARLY STOPPING И CHECKPOINT
# ==========================================
NUM_EPOCHS = 10
best_val_loss = float('inf')
patience_early_stop = 3 # Сколько эпох терпеть рост Val Loss перед принудительной остановкой
early_stop_counter = 0

print("\n🎬 Старт безопасного дообучения 3D-ResNet18...")
for epoch in range(NUM_EPOCHS):
    model_3d.train()
    train_loss, correct_train, total_train = 0.0, 0, 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        outputs = model_3d(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_X.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += batch_y.size(0)
        correct_train += (predicted == batch_y).sum().item()

    # Валидация (аугментация выключена автоматически)
    model_3d.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model_3d(batch_X)
            loss = criterion(outputs, batch_y)

            val_loss += loss.item() * batch_X.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += batch_y.size(0)
            correct_val += (predicted == batch_y).sum().item()

    train_loss_avg = train_loss / total_train
    val_loss_avg = val_loss / total_val
    train_acc = correct_train / total_train
    val_acc = correct_val / total_val

    scheduler.step(val_loss_avg)

    # CHECKPOINT: Сохраняем строго лучшую точку схождения по Val Loss
    if val_loss_avg < best_val_loss:
        best_val_loss = val_loss_avg
        torch.save(model_3d.state_dict(), "best_3d_resnet.pth")
        early_stop_counter = 0
        status = "🔥 [Сохранено! Лучший Val Loss]"
    else:
        early_stop_counter += 1
        status = f"⚠️ [Не улучшилось. Шаг до остановки: {patience_early_stop - early_stop_counter}]"

    print(f"Эпоха [{epoch+1}/{NUM_EPOCHS}] | "
          f"Loss Train/Val: {train_loss_avg:.4f} / {val_loss_avg:.4f} | "
          f"Acc Train/Val: {train_acc:.2%} / {val_acc:.2%} {status}")

    # EARLY STOPPING: Принудительный выход, если Val Loss растет 3 эпохи подряд
    if early_stop_counter >= patience_early_stop:
        print(f"\n🛑 Ранняя остановка (Early Stopping) сработала на эпохе {epoch+1}!")
        break

print("\n🎉 Пайплайн завершен. Финальные идеальные веса лежат в: best_3d_resnet.pth")


Используем устройство для защищенной 3D-ResNet18: cuda
🧬 Рассчитанные веса для балансировки Loss функции: [1.80620155 2.05934343 0.51000625]
📥 Загрузка базовой r3d_18...

🎬 Старт безопасного дообучения 3D-ResNet18...
Эпоха [1/10] | Loss Train/Val: 0.5274 / 0.3714 | Acc Train/Val: 75.08% / 88.38% 🔥 [Сохранено! Лучший Val Loss]
Эпоха [2/10] | Loss Train/Val: 0.1991 / 0.3670 | Acc Train/Val: 93.06% / 91.44% 🔥 [Сохранено! Лучший Val Loss]
Эпоха [3/10] | Loss Train/Val: 0.0954 / 0.3298 | Acc Train/Val: 97.61% / 92.66% 🔥 [Сохранено! Лучший Val Loss]
Эпоха [4/10] | Loss Train/Val: 0.0547 / 0.3489 | Acc Train/Val: 98.69% / 91.74% ⚠️ [Не улучшилось. Шаг до остановки: 2]
Эпоха [5/10] | Loss Train/Val: 0.0447 / 0.4486 | Acc Train/Val: 98.84% / 89.60% ⚠️ [Не улучшилось. Шаг до остановки: 1]
Эпоха [6/10] | Loss Train/Val: 0.0383 / 0.4787 | Acc Train/Val: 99.00% / 91.13% ⚠️ [Не улучшилось. Шаг до остановки: 0]

🛑 Ранняя остановка (Early Stopping) сработала на эпохе 6!

🎉 Пайплайн завершен. Финальные